설명 가능한 딥러닝 방식을 구현하게 해 주는 tf-explain 라이브러리와 이를 돕는 OpenCV 라이브러리가 필요하다.  

In [1]:
# 코랩에서는 OpenCV 라이브러리를 기본적으로 제공하므로 tf-explain 라이브러리만 설치한다.
!pip install tf-explain

Gradient CAM, 폐쇄성 민감도 방식 함수를 불러온다.

In [2]:
from tf_explain.core.grad_cam import GradCAM
from tf_explain.core.occlusion_sensitivity import OcclusionSensitivity

Imagenet 학습 모델을 불러와 우리의 모델로 사용한다.

In [3]:
from tensorflow.keras.applications import VGG16

model = VGG16(weights="imagenet", include_top=True)

그레이디언트 CAM을 실행한다.

```
explainer = GradCAM()                               # 그레이디언트 CAM 알고리즘 선택
output = explainer.explain(데이터, 모델, 클래스)    # 그레이디언트 CAM 실행
explainer.save(output, 저장될 폴더, 저장될 이름)    # 실행 후 저장될 위치와 이름

```

먼저 GradCAM() 함수를 expainer 인스턴스에 할당했다.  
XAI를 실행하는 explain() 함수와 이를 저장하는 save() 함수가 있다. explain() 함수 안에는 데이터, 모델, 이미지넷의 클래스 번호가 들어간다. save() 함수 안에는 XAI를 실행한 결과, 저장될 폴더, 그리고 저장될 이름이 들어간다.

오클루전 방식은 다음과 같이 실행한다.


```
explainer = OcclusionSensitivity() # 오클루전 알고리즘 선택
# 패치 크기 설정이 추가됨
output = explainer.explain(데이터, 모델, 클래스, 패치 크기)
explainer.save(output, 저장될 폴더, 저장될 이름)
```

그레이디언트 CAM 방식과의 차이점은 explain() 함수 안의 인자로 패치 크기가 들어간다는 것이다.  
패치 크기를 크게 잡으면 조금 더 넓은 범위의 결과가 나오고, 작게 잡으면 조금 더 세밀한 부분을 가리키는 결과가 나온다.

실습| 설명 가능한 딥러닝

In [4]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import VGG16

# XAI 알고리즘을 불러오는 부분이다.
from tf_explain.core.grad_cam import GradCAM
from tf_explain.core.occlusion_sensitivity import OcclusionSensitivity

# 이미지를 불러와 보여 주는 데 쓰는 라이브러리를 불러온다.
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# 깃허브에 준비된 데이터를 가져온다.
!git clone https://github.com/taehojo/data.git

# 원본 이미지가 들어갈 리스트를 만든다.
images_originals = []

# 원본 이미지가 저장된 폴더에서 하나씩 불러와 리스트에 넣는다.
for img_path in sorted(glob.glob('./data/img/*_0.jpg')):
    images_originals.append(mpimg.imread(img_path))

# 코랩에서 보여 줄 이미지의 크기
plt.figure(figsize=(20, 20))

# 원본 이미지를 코랩에서 보이게 하기
for i, image_o in enumerate(images_originals):
    plt.subplot(5, 5, i + 1)
    plt.imshow(image_o)

# 사전에 학습된 딥러닝 모델 불러오기
model = VGG16(weights="imagenet", include_top=True)

# 원본 이미지 이름과 Imagenet에서 해당 이미지 인덱스
input_list = ["maltese", "persian_cat", "squirrel_monkey", "grand_piano", "yawl"]
imagenet_index = ["153", "283", "382", "579", "914"]

# 그레이디언트 CAM 알고리즘 선택
explainer = GradCAM()

# 그레이디언트 CAM 알고리즘이 적용된 이미지가 들어갈 빈 리스트 만들기
images_cams = []

# 그레이디언트 CAM 알고리즘 실행
for l, i in zip(input_list, imagenet_index):
    # 이미지를 불러오고 내부에서 처리될 이미지의 크기를 설정한다.
    img = load_img('./data/img/{}_0.jpg'.format(l), target_size=(224, 224))
    img = img_to_array(img) # 이미지를 넘파이 배열로 바꾼다.
    data = ([img], None)
    # 그레이디언트 CAM이 실행되는 부분
    grid = explainer.explain(data, model, int(i))
    # 실행 후 저장되는 이름이다.
    explainer.save(grid, ".", "./data/img/{}_cam.jpg".format(l))

# 그레이디언트 CAM 알고리즘이 적용된 이미지를 불러오는 부분의 시작이다.
plt.figure(figsize=(20, 20))

for img_path in sorted(glob.glob('./data/img/*_cam.jpg')):
    images_cams.append(mpimg.imread(img_path))

for i, image_c in enumerate(images_cams):
    plt.subplot(5, 5, i+1)
    plt.imshow(image_c)

# 오클루전 알고리즘 선택
explainer = OcclusionSensitivity()

# 알고리즘이 적용된 이미지가 들어갈 빈 리스트 만들기
images_occ1s = []

# 패치 크기를 정한다.
patch_size = 40

# 오클루전 알고리즘 실행
for l, i in zip(input_list, imagenet_index):
    img = load_img('./data/img/{}_0.jpg'.format(l), target_size=(224, 224))
    img = img_to_array(img)
    data = ([img], None)
    # 패치 크기 설정이 추가된다.
    grid = explainer.explain(data, model, int(i), patch_size)
    explainer.save(grid, ".", './data/img/{}_occ1.jpg'.format(l))

# 오클루전 알고리즘이 적용된 이미지를 불러오는 부분의 시작이다.
plt.figure(figsize=(20, 20))

for img_path in sorted(glob.glob('./data/img/*_occ1.jpg')):
    images_occ1s.append(mpimg.imread(img_path))

for i, image in enumerate(images_occ1s):
    plt.subplot(5, 5, i+1)
    plt.imshow(image)

Output hidden; open in https://colab.research.google.com to view.